In [ ]:
import random
import joblib
from nn_models   import *
from nn_data     import *
from nn_training import *
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from decimal import Decimal, ROUND_FLOOR

In [ ]:
DATA_PATH    = '/path'
BEST_HP_PATH = '/path'

### 1. Load data

In [ ]:
features = KEY_FEATURES + CONTROL_FEATURES
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_test  = pd.read_csv(DATA_PATH + 'test.csv')
df_test  = df_test.dropna(subset='pm25')

X_train = np.array(df_train[features])
y_train = np.array(df_train['pm25_filled'])
X_test  = np.array(df_test[features])
y_test  = np.array(df_test['pm25'])

### 2. Standardize data

In [ ]:
loaded_study = joblib.load(BEST_HP_PATH)
best = loaded_study.best_params

batch_size = best["batch_size"]
n_features = len(features)

# Fit statistic on train only
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

feature_scaler.fit(X_train)
target_scaler.fit(y_train.reshape(-1, 1))

# Transform the data
train_dataset = MLPDataset(X_train, y_train, feature_scaler, target_scaler)
test_dataset  = MLPDataset(X_test,  y_test,  feature_scaler, target_scaler)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

### 3. Train the model

In [4]:
best_trial = loaded_study.best_trial
best_epoch = best_trial.user_attrs["best_epoch"]

In [ ]:
seed_everything()

model     = MLP(input_size   = len(features), 
                h1           = best_trial.user_attrs['h1'], 
                h2           = best_trial.user_attrs['h2'], 
                hout         = best_trial.user_attrs['hout'], 
                drop_rate    = best["drop_rate"], 
                output_size  = 1)

optimizer = torch.optim.Adam(model.parameters(), lr = best['lr'])
criterion = nn.MSELoss()
epochs    = best_epoch

train_losses = train_MLP(model, train_loader, criterion, optimizer, epochs)  # X: [B, F]

### 4. Collect predictions

In [ ]:
df_test_pred   = collect_MLP_preds(test_dataset.X,  
                                   df_test, 
                                   model, 
                                   target_scaler)